In [13]:
import numpy as np

def find_multipliers(fz: float, fd: float, err_rel_ref: float, Nmin: int, Nmax: int):
    R = fd / fz
    
    if R < 2:
        raise ValueError("fz must be < fd/2 for a valid solution.")
    
    err_abs_ref = fz*err_rel_ref
    Mmax = int(R / 2)
    N_vals = np.arange(Nmin, Nmax + 1, dtype=float)
    
    N_arr = np.empty(Mmax, dtype=np.int64)
    M_arr = np.empty(Mmax, dtype=np.int64)
    L_arr = np.empty(Mmax, dtype=np.int64)
    fz_approx_arr = np.empty(Mmax)
    err_abs_arr = np.empty(Mmax)
    err_rel_arr = np.empty(Mmax)
    tol_flag_arr = np.empty(Mmax, dtype=bool)
    
    for mdx, M in enumerate(range(1, Mmax+1)):
        L = np.floor(M * N_vals / R)
        # In-place clamp.
        np.maximum(L, 1, out=L)
        
        R_approx = M * N_vals / L
        fz_approx = fd / R_approx
        err_abs = np.abs(fz - fz_approx)
        
        valid_idx = np.nonzero(err_abs < err_abs_ref)[0]
        
        if valid_idx.size > 0:
            pos = valid_idx[0]
            tol_flag_arr[mdx] = True
        else:
            pos = np.argmin(err_abs)
            tol_flag_arr[mdx] = False

        
        N_arr[mdx] = int(N_vals[pos])
        M_arr[mdx] = M
        L_arr[mdx] = int(L[pos])
        fz_approx_arr[mdx] = fz_approx[pos]
        err_abs_arr[mdx] = err_abs[pos]
        err_rel_arr[mdx] = err_abs[pos] / fz_approx[pos]
    
    if np.any(tol_flag_arr):
        valid_mdx = np.nonzero(tol_flag_arr)[0]
        best_mdx = valid_mdx[np.argmin(N_arr[valid_mdx])]
    else:
        best_mdx = np.argmin(err_abs_arr)
        
    return (int(N_arr[best_mdx]),
            int(M_arr[best_mdx]),
            int(L_arr[best_mdx])), \
           (float(fz_approx_arr[best_mdx]),
            float(err_abs_arr[best_mdx]),
            float(err_rel_arr[best_mdx]),
            bool(tol_flag_arr[best_mdx]))

# Проверка на одной частоте

In [14]:
fd = 1e5
fz = 22100
err_rel_ref = 1e-3

Nmin, Nmax = [10, 1214]

(N, M, L), debug_info = find_multipliers(fz=fz, fd=fd, err_rel_ref=err_rel_ref, Nmin=Nmin, Nmax=Nmax)
print(f"N: {N}, M: {M}, L: {L}")
print(debug_info)

N: 43, M: 2, L: 19
(22093.023255813954, 6.976744186045835, 0.0003157894736841799, True)


# Проверка в диапазоне частот

In [21]:
fd = 1e5
err_rel_ref = 1e-3

fmin = 10
fmax = 50e3
K = 20
freq = np.logspace(np.log10(fmin), np.log10(fmax), K)

for f in freq:
    (N, M, L), debug_info = find_multipliers(fz=f, fd=fd, err_rel_ref=err_rel_ref, Nmin=Nmin, Nmax=Nmax)
    print(f"N: {N}, M: {M}, L: {L} | fz: {f:.3f}, fz_approx: {debug_info[0]:.3f}, {debug_info[-1]}")

N: 10, M: 1000, L: 1 | fz: 10.000, fz_approx: 10.000, True
N: 10, M: 639, L: 1 | fz: 15.656, fz_approx: 15.649, True
N: 10, M: 408, L: 1 | fz: 24.511, fz_approx: 24.510, True
N: 10, M: 782, L: 3 | fz: 38.375, fz_approx: 38.363, True
N: 10, M: 333, L: 2 | fz: 60.080, fz_approx: 60.060, True
N: 10, M: 319, L: 3 | fz: 94.062, fz_approx: 94.044, True
N: 11, M: 247, L: 4 | fz: 147.264, fz_approx: 147.221, True
N: 14, M: 31, L: 1 | fz: 230.557, fz_approx: 230.415, True
N: 11, M: 126, L: 5 | fz: 360.962, fz_approx: 360.750, True
N: 12, M: 59, L: 4 | fz: 565.124, fz_approx: 564.972, True
N: 29, M: 39, L: 10 | fz: 884.762, fz_approx: 884.173, True
N: 17, M: 17, L: 4 | fz: 1385.189, fz_approx: 1384.083, True
N: 17, M: 19, L: 7 | fz: 2168.661, fz_approx: 2167.183, True
N: 27, M: 12, L: 11 | fz: 3395.269, fz_approx: 3395.062, True
N: 23, M: 9, L: 11 | fz: 5315.656, fz_approx: 5314.010, True
N: 89, M: 5, L: 37 | fz: 8322.225, fz_approx: 8314.607, True
N: 64, M: 3, L: 25 | fz: 13029.331, fz_approx: 